[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-2-generative-ai/adv-04-rope-moe-online-softmax-parameter-accounting.ipynb)

# Advanced Discussion 4 — RoPE, expert routing, online softmax and parameter accounting

RoPE from scratch and its relative-position property, a toy mixture-of-experts with a balance loss, exact block-wise attention, and parameter and KV-cache counts checked against published model sizes.

**Optional advanced-discussion lab** for Generative AI and Large Language Models with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier; CPU is enough unless a note says otherwise.

CPU only; runs in about two minutes.

In [ ]:
%pip install -q torch numpy

In [ ]:
import math, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0); np.random.seed(0)

## 1. RoPE from scratch: rotate each pair of dimensions by an angle proportional to the position

In [ ]:
def rope(x, pos, base=10000.0):
    """x: (..., d) with d even; pos: (...,) integer positions. Pair (2i, 2i+1) is rotated by pos * base^(-2i/d)."""
    d = x.shape[-1]; inv = base ** (-torch.arange(0, d, 2).float() / d); ang = pos.unsqueeze(-1).float() * inv
    x1, x2 = x[..., 0::2], x[..., 1::2]; c, s = ang.cos(), ang.sin()
    out = torch.empty_like(x); out[..., 0::2] = x1 * c - x2 * s; out[..., 1::2] = x1 * s + x2 * c; return out
d = 64; q = torch.randn(d); k = torch.randn(d)
def score(m, n): return (rope(q, torch.tensor(m)) @ rope(k, torch.tensor(n))).item()
print("RoPE: the score between query at position m and key at position n depends only on n - m")
for m, n in ((5, 2), (105, 102), (5005, 5002), (50005, 50002)):
    print("  m=%5d n=%5d (distance %d): score %.5f" % (m, n, m - n, score(m, n)))
print("  a learned absolute position added to q and k would give different scores for those four pairs")
dists = [0, 1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]
print("  same content at distance: " + "  ".join("%d:%.1f" % (t, score(t, 0)) for t in dists))
# wavelengths of the rotation frequencies, and how many complete a full turn inside a given context length
wl = 2 * math.pi * 10000 ** (torch.arange(0, d, 2).float() / d)
print("\nwavelengths (in tokens) of the %d rotation frequencies for d=64, base 10000: shortest %.1f, median %.0f, longest %.0f" % (d // 2, wl.min(), wl.median(), wl.max()))
for L in (2048, 4096, 8192, 32768):
    print("  context %6d tokens: %2d of %d frequencies complete at least one full turn" % (L, int((wl <= L).sum()), d // 2))
# extending context: position interpolation squeezes positions back into the trained range
trained = 4096
for new in (8192, 16384, 32768):
    scale = trained / new; novel = int((wl > trained).sum())                          # frequencies that never completed a full turn while training: their angles beyond that are unseen values
    print("  trained on %d, run at %d: %d of %d frequencies now see angles they never saw in training (those with wavelength > %d); with position interpolation (positions x %.3f) that becomes 0, at the price of every distance being compressed %.0fx" % (trained, new, novel, d // 2, trained, scale, 1 / scale))
print("  (whether a trained model then reads the compressed positions well is an empirical question; published recipes fine-tune briefly after rescaling)")

## 2. mixture of experts: routing, load balance and the auxiliary loss

In [ ]:
E, K, DIN, N = 8, 2, 16, 2048
centers = torch.randn(4, DIN) * 3; cluster_map = torch.randn(4, DIN, DIN) * 0.3          # 4 kinds of tokens, each needs a different transformation
def batch(n=N):
    c = torch.randint(0, 4, (n,)); x = centers[c] + torch.randn(n, DIN) * 0.5; y = torch.einsum("bi,bij->bj", x, cluster_map[c]); return x, y, c
class MoE(nn.Module):
    def __init__(s, alpha):
        super().__init__(); s.router = nn.Linear(DIN, E); s.experts = nn.ModuleList([nn.Sequential(nn.Linear(DIN, 32), nn.GELU(), nn.Linear(32, DIN)) for _ in range(E)]); s.alpha = alpha
    def forward(s, x):
        logits = s.router(x); probs = F.softmax(logits, -1); topv, topi = probs.topk(K, -1); gates = topv / topv.sum(-1, keepdim=True)
        out = torch.zeros_like(x)
        for e in range(E):
            sel = (topi == e); rows = sel.any(-1)
            if rows.any(): out[rows] += (gates * sel)[rows].sum(-1, keepdim=True) * s.experts[e](x[rows])
        frac = F.one_hot(topi, E).float().sum(1).mean(0) / K                          # fraction of tokens routed to each expert (top-k counted)
        aux = E * (frac * probs.mean(0)).sum()                                        # Switch-style balance loss: minimal when load is uniform
        return out, aux, frac
def train_moe(alpha, steps=600, seed=0):
    torch.manual_seed(seed); m = MoE(alpha); opt = torch.optim.Adam(m.parameters(), 3e-3)
    for _ in range(steps):
        x, y, _ = batch(); out, aux, _ = m(x); loss = F.mse_loss(out, y) + alpha * aux; opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad(): x, y, c = batch(8192); out, aux, frac = m(x); mse = F.mse_loss(out, y).item()
    return mse, frac, m, x
print("\nmixture of experts: 8 experts, top-2 routing, 4 kinds of tokens; 600 steps")
print("aux-loss weight   test MSE   load per expert (share of tokens)                         max/mean load   experts under 2% load   tokens dropped at capacity 1.25x")
for alpha in (0.0, 0.01, 0.1):
    res = [train_moe(alpha, seed=s) for s in range(3)]
    mse = np.mean([r[0] for r in res]); fr = torch.stack([r[1] for r in res]).mean(0)
    # capacity: each expert can take 1.25 x the mean number of token-slots; overflow is dropped
    dropped = []
    for r in res:
        _, frac, m, x = r
        with torch.no_grad():
            probs = F.softmax(m.router(x), -1); topi = probs.topk(K, -1).indices
        counts = F.one_hot(topi, E).float().sum((0, 1)); cap = 1.25 * counts.sum() / E; dropped.append(((counts - cap).clamp(min=0).sum() / counts.sum()).item())
    print("  %-12.2f    %.4f    %s    %5.2f           %d                       %.1f%%" % (alpha, mse, " ".join("%.2f" % f for f in fr / fr.sum()), (fr / fr.mean()).max().item(), int((fr / fr.sum() < 0.02).sum()), 100 * np.mean(dropped)))

## 3. FlashAttention's idea: exact attention computed block by block with an online softmax

In [ ]:
def attention_exact(Q, K_, V): return F.softmax(Q @ K_.T / math.sqrt(Q.shape[-1]), -1) @ V
def attention_online(Q, K_, V, block=128):
    n, dd = Q.shape; out = torch.zeros(n, V.shape[1]); m = torch.full((n, 1), -float("inf")); l = torch.zeros(n, 1); peak_elems = 0
    for j in range(0, K_.shape[0], block):
        Kj, Vj = K_[j:j + block], V[j:j + block]; s = Q @ Kj.T / math.sqrt(dd); peak_elems = max(peak_elems, s.numel())
        m_new = torch.maximum(m, s.max(-1, keepdim=True).values); p = torch.exp(s - m_new); scale = torch.exp(m - m_new)
        l = l * scale + p.sum(-1, keepdim=True); out = out * scale + p @ Vj; m = m_new
    return out / l, peak_elems
n = 1024; Q, K_, V = torch.randn(n, 64), torch.randn(n, 64), torch.randn(n, 64)
ref = attention_exact(Q, K_, V)
print("\nonline-softmax attention vs exact attention, n=%d:" % n)
for b in (32, 128, 512):
    o, peak = attention_online(Q, K_, V, b); print("  block %3d: max |difference| %.1e; largest score tile held at once %d entries (full matrix %d, %.0f%%)" % (b, (o - ref).abs().max().item(), peak, n * n, 100 * peak / (n * n)))
big = Q * 40; o, _ = attention_online(big, K_ * 40, V); print("  with scores ~ +/-1,000 the online version still matches exact softmax: max difference %.1e" % (o - attention_exact(big, K_ * 40, V)).abs().max().item())
s = torch.tensor([100.0, 99.0, 98.0], dtype=torch.float16); naive = torch.exp(s) / torch.exp(s).sum(); stable = torch.exp(s - s.max()) / torch.exp(s - s.max()).sum()
print("  float16 softmax of [100, 99, 98]: naive exp/sum -> %s ; subtract the max first -> %s" % (naive.tolist(), [round(v, 3) for v in stable.tolist()]))

## 4. parameter and KV-cache accounting for a modern decoder, checked against published totals

In [ ]:
def count(d, layers, heads, kv_heads, ffn, vocab, experts=1, active=1, tied=False, gated=True):
    hd = d // heads; attn = d * (heads * hd) + 2 * d * (kv_heads * hd) + (heads * hd) * d
    ffn_p = (3 if gated else 2) * d * ffn; norms = 2 * d
    per_layer_total = attn + experts * ffn_p + (d * experts if experts > 1 else 0) + norms
    per_layer_active = attn + active * ffn_p + (d * experts if experts > 1 else 0) + norms
    emb = vocab * d * (1 if tied else 2)
    return layers * per_layer_total + emb + d, layers * per_layer_active + emb + d, 2 * layers * kv_heads * hd * 2
cfgs = [("Llama-2 7B", dict(d=4096, layers=32, heads=32, kv_heads=32, ffn=11008, vocab=32000)),
        ("Llama-2 70B", dict(d=8192, layers=80, heads=64, kv_heads=8, ffn=28672, vocab=32000)),
        ("Mixtral 8x7B", dict(d=4096, layers=32, heads=32, kv_heads=8, ffn=14336, vocab=32000, experts=8, active=2))]
print("\nparameters computed from the architecture configuration (published: 6.74B, ~69B, 46.7B total / 12.9B active)")
print("model           total params   active per token   KV cache per token (fp16)   KV cache for 32k tokens")
for name, c in cfgs:
    t, a, kv = count(**c); print("%-14s  %8.2fB       %8.2fB          %7.1f KB                   %5.1f GB" % (name, t / 1e9, a / 1e9, kv / 1024, kv * 32768 / 1e9))

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.